# Implied Volatility Surface: SVI Calibration on SPY Options

In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import os

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../results/tables', exist_ok=True)

from src.data_utils import (
    fetch_options_chain, load_cached_data,
    generate_synthetic_options, fetch_underlying_history
)
from src.implied_vol import compute_all_ivs
from src.vol_surface import (
    build_vol_surface, svi_implied_vol, svi_total_variance,
    check_butterfly_arbitrage, check_calendar_arbitrage,
    interpolate_surface, compute_smile_metrics
)
from src.realized_vol import close_to_close, parkinson, yang_zhang, variance_risk_premium
from src import config

print('All imports successful.')

# 1. Fetch SPY Options Data

In [ ]:
data_source = 'unknown'

try:
    options_df, spot, r, q = fetch_options_chain('SPY', save_raw=True)
    data_source = 'live (yfinance)'
    print(f'Fetched live SPY options data: {len(options_df)} rows')
except Exception as e1:
    print(f'Live fetch failed: {e1}')
    try:
        options_df, spot, r, q = load_cached_data('SPY')
        data_source = 'cached'
        print(f'Loaded cached SPY options data: {len(options_df)} rows')
    except Exception as e2:
        print(f'Cache load failed: {e2}')
        print('Generating synthetic options data...')
        options_df, spot, r, q = generate_synthetic_options(spot=585.0, n_expirations=6)
        data_source = 'synthetic'
        print(f'Generated synthetic data: {len(options_df)} rows')

print(f'\nData source: {data_source}')
print(f'Spot: ${spot:.2f}')
print(f'Risk-free rate: {r:.4f}')
print(f'Dividend yield: {q:.4f}')

# 2. Data Summary

In [ ]:
print(f'Total contracts: {len(options_df)}')
print(f'Calls: {(options_df["option_type"] == "call").sum()}')
print(f'Puts: {(options_df["option_type"] == "put").sum()}')

if 'expiration' in options_df.columns:
    n_exp = options_df['expiration'].nunique()
else:
    n_exp = options_df['T'].nunique()
print(f'Unique expirations: {n_exp}')
print(f'Strike range: [{options_df["strike"].min():.2f}, {options_df["strike"].max():.2f}]')
print(f'Moneyness range: [{options_df["moneyness"].min():.3f}, {options_df["moneyness"].max():.3f}]')
print(f'T range: [{options_df["T"].min():.4f}, {options_df["T"].max():.4f}] years')
print(f'\nColumns: {list(options_df.columns)}')

# 3. Compute Implied Volatilities

In [ ]:
# compute_all_ivs expects columns: mid_price, strike, T, option_type
iv_df = compute_all_ivs(options_df, spot=spot, r=r, q=q)

print(f'\nIV computation complete: {len(iv_df)} valid options')
print(f'IV range: [{iv_df["computed_iv"].min():.4f}, {iv_df["computed_iv"].max():.4f}]')
print(f'Mean IV: {iv_df["computed_iv"].mean():.4f}')

# 4. Raw Market IVs

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

calls = iv_df[iv_df['option_type'] == 'call']
puts = iv_df[iv_df['option_type'] == 'put']

sc_calls = ax.scatter(calls['moneyness'], calls['computed_iv'] * 100,
                      c=calls['T'] * 365, cmap='Blues', alpha=0.6, s=15, label='Calls')
sc_puts = ax.scatter(puts['moneyness'], puts['computed_iv'] * 100,
                     c=puts['T'] * 365, cmap='Reds', alpha=0.6, s=15, label='Puts')

ax.axvline(x=1.0, color='gray', linestyle='--', alpha=0.5, label='ATM')
ax.set_xlabel('Moneyness (K/S)', fontsize=12)
ax.set_ylabel('Implied Volatility (%)', fontsize=12)
ax.set_title('Raw Market Implied Volatilities (SPY)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

cbar = plt.colorbar(sc_calls, ax=ax)
cbar.set_label('Days to Expiry', fontsize=10)

plt.tight_layout()
plt.savefig('../results/figures/raw_market_ivs.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/raw_market_ivs.png')

# 5. Realized Volatility (3 Estimators)

In [ ]:
# Fetch underlying history for realized vol
try:
    hist_df = fetch_underlying_history('SPY', period='2y')
    print(f'Fetched SPY history: {len(hist_df)} trading days')
except Exception as e:
    print(f'Could not fetch history: {e}')
    # Create synthetic history
    dates = pd.bdate_range(end=pd.Timestamp.now(), periods=504)
    np.random.seed(42)
    returns = np.random.normal(0.0004, 0.012, len(dates))
    prices = spot * np.exp(np.cumsum(returns))
    high_mult = 1 + np.abs(np.random.normal(0, 0.005, len(dates)))
    low_mult = 1 - np.abs(np.random.normal(0, 0.005, len(dates)))
    hist_df = pd.DataFrame({
        'Open': prices * (1 + np.random.normal(0, 0.002, len(dates))),
        'High': prices * high_mult,
        'Low': prices * low_mult,
        'Close': prices,
    }, index=dates)
    print(f'Generated synthetic history: {len(hist_df)} days')

In [ ]:
window = 21  # 1 month rolling window

rv_cc = close_to_close(hist_df['Close'], window=window)
rv_park = parkinson(hist_df['High'], hist_df['Low'], window=window)
rv_yz = yang_zhang(hist_df['Open'], hist_df['High'], hist_df['Low'], hist_df['Close'], window=window)

# ATM implied vol (approximate from our data)
atm_mask = (iv_df['moneyness'] >= 0.98) & (iv_df['moneyness'] <= 1.02)
atm_iv_mean = iv_df.loc[atm_mask, 'computed_iv'].mean() if atm_mask.sum() > 0 else 0.18

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(rv_cc.index[-252:], rv_cc.iloc[-252:] * 100, color='steelblue', linewidth=1.5,
        label='Close-to-Close', alpha=0.9)
ax.plot(rv_park.index[-252:], rv_park.iloc[-252:] * 100, color='darkorange', linewidth=1.5,
        label='Parkinson', alpha=0.9)
ax.plot(rv_yz.index[-252:], rv_yz.iloc[-252:] * 100, color='forestgreen', linewidth=1.5,
        label='Yang-Zhang', alpha=0.9)
ax.axhline(y=atm_iv_mean * 100, color='crimson', linestyle='--', linewidth=2,
           label=f'ATM IV = {atm_iv_mean*100:.1f}%')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Annualized Volatility (%)', fontsize=12)
ax.set_title('Realized Volatility Estimators vs Implied Volatility (SPY)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')

plt.tight_layout()
plt.savefig('../results/figures/realized_vs_implied.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/realized_vs_implied.png')

# 6. Variance Risk Premium

In [ ]:
rv_current = float(rv_cc.dropna().iloc[-1])
vrp = variance_risk_premium(rv_current, atm_iv_mean)

print('Variance Risk Premium (VRP)')
print('=' * 40)
print(f'ATM Implied Vol:       {atm_iv_mean*100:.2f}%')
print(f'Realized Vol (21d CC): {rv_current*100:.2f}%')
print(f'VRP = IV - RV:         {vrp*100:+.2f}%')
print(f'\nInterpretation: {"Options are priced ABOVE realized -- sellers earn premium" if vrp > 0 else "Options are priced BELOW realized -- unusual regime"}')

# 7. SVI Surface Calibration

In [ ]:
surface_params = build_vol_surface(iv_df, spot=spot, r=r, q=q)

print(f'\nCalibrated {len(surface_params)} expiration slices.')

# Save calibration summary
summary_rows = []
for T_key, params in sorted(surface_params.items()):
    summary_rows.append({
        'T': f'{T_key:.4f}',
        'Days': f'{T_key*365:.0f}',
        'a': f'{params["a"]:.6f}',
        'b': f'{params["b"]:.6f}',
        'rho': f'{params["rho"]:.6f}',
        'm': f'{params["m"]:.6f}',
        'sigma': f'{params["sigma"]:.6f}',
        'RMSE_bps': f'{params["rmse_bps"]:.1f}',
        'R2': f'{params["r_squared"]:.6f}',
        'BF_arb_free': params.get('butterfly_arb_free', 'N/A'),
    })

cal_summary_df = pd.DataFrame(summary_rows)
print(cal_summary_df.to_string(index=False))

cal_summary_df.to_csv('../results/tables/svi_calibration_summary.csv', index=False)
print('\nSaved: results/tables/svi_calibration_summary.csv')

# 8. Vol Smile Plots

In [ ]:
sorted_T = sorted(surface_params.keys())
n_slices = len(sorted_T)
ncols = min(3, n_slices)
nrows = (n_slices + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
if n_slices == 1:
    axes = np.array([axes])
axes = np.atleast_2d(axes)

k_fine = np.linspace(-0.3, 0.3, 200)

for idx, T_key in enumerate(sorted_T):
    ax = axes[idx // ncols, idx % ncols]
    params = surface_params[T_key]
    forward = params['forward']

    # SVI fitted curve
    iv_fit = svi_implied_vol(k_fine, T_key, params['a'], params['b'],
                             params['rho'], params['m'], params['sigma'])
    ax.plot(k_fine, iv_fit * 100, color='crimson', linewidth=2, label='SVI Fit')

    # Market data points for this slice
    if 'expiration' in iv_df.columns:
        slice_data = iv_df[iv_df['T'].round(4) == round(T_key, 4)]
    else:
        slice_data = iv_df[np.abs(iv_df['T'] - T_key) < 0.001]

    if len(slice_data) > 0:
        k_market = np.log(slice_data['strike'].values / forward)
        iv_market = slice_data['computed_iv'].values
        ax.scatter(k_market, iv_market * 100, color='steelblue', s=20, alpha=0.7,
                   zorder=5, label='Market')

    ax.set_xlabel('Log-Moneyness ln(K/F)', fontsize=10)
    ax.set_ylabel('IV (%)', fontsize=10)
    ax.set_title(f'T = {T_key*365:.0f}d (RMSE={params["rmse_bps"]:.0f}bps)', fontsize=11)
    ax.legend(fontsize=9)
    ax.set_xlim(-0.3, 0.3)

# Hide unused subplots
for idx in range(n_slices, nrows * ncols):
    axes[idx // ncols, idx % ncols].set_visible(False)

plt.suptitle('SVI Vol Smiles by Expiration', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/vol_smiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/vol_smiles.png')

# 9. 3D Volatility Surface

In [ ]:
# Build interpolated surface
strike_grid = np.linspace(spot * 0.85, spot * 1.15, 80)
T_grid = np.linspace(min(sorted_T), max(sorted_T), 50)

iv_surface = interpolate_surface(surface_params, strike_grid, T_grid, spot, r, q)

# Moneyness grid for plotting
moneyness_grid = strike_grid / spot
M_mesh, T_mesh = np.meshgrid(moneyness_grid, T_grid * 365)

fig = plt.figure(figsize=(14, 9))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(
    M_mesh, T_mesh, iv_surface * 100,
    cmap=cm.coolwarm, edgecolor='none', alpha=0.85
)

ax.set_xlabel('Moneyness (K/S)', fontsize=11, labelpad=12)
ax.set_ylabel('Days to Expiry', fontsize=11, labelpad=12)
ax.set_zlabel('Implied Volatility (%)', fontsize=11, labelpad=10)
ax.set_title('SPY Implied Volatility Surface (SVI)', fontsize=14, fontweight='bold')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, pad=0.1, label='IV (%)')
ax.view_init(elev=25, azim=-60)

plt.tight_layout()
plt.savefig('../results/figures/vol_surface_3d.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/vol_surface_3d.png')

# 10. ATM Term Structure

In [ ]:
atm_ivs = []
for T_key in sorted_T:
    params = surface_params[T_key]
    atm_iv_val = float(svi_implied_vol(np.array([0.0]), T_key,
                       params['a'], params['b'], params['rho'],
                       params['m'], params['sigma'])[0])
    atm_ivs.append(atm_iv_val)

fig, ax = plt.subplots(figsize=(10, 6))
days_to_exp = [t * 365 for t in sorted_T]
ax.plot(days_to_exp, [iv * 100 for iv in atm_ivs], 'o-', color='steelblue',
        linewidth=2, markersize=8)

ax.set_xlabel('Days to Expiry', fontsize=12)
ax.set_ylabel('ATM Implied Volatility (%)', fontsize=12)
ax.set_title('ATM Volatility Term Structure', fontsize=13, fontweight='bold')

# Annotate each point
for d, iv_val in zip(days_to_exp, atm_ivs):
    ax.annotate(f'{iv_val*100:.1f}%', (d, iv_val * 100),
                textcoords='offset points', xytext=(0, 12),
                ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../results/figures/vol_term_structure.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/vol_term_structure.png')

# 11. Skew Term Structure

In [ ]:
skew_metrics = []
for T_key in sorted_T:
    params = surface_params[T_key]
    metrics = compute_smile_metrics(params, params['forward'], T_key)
    metrics['T'] = T_key
    metrics['days'] = T_key * 365
    skew_metrics.append(metrics)

skew_df = pd.DataFrame(skew_metrics)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Skew slope
ax = axes[0]
ax.plot(skew_df['days'], skew_df['skew_slope'], 'o-', color='crimson', linewidth=2, markersize=7)
ax.set_xlabel('Days to Expiry', fontsize=11)
ax.set_ylabel('Skew Slope (dIV/dk)', fontsize=11)
ax.set_title('ATM Skew Slope', fontsize=12, fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# 25d Risk Reversal
ax = axes[1]
ax.plot(skew_df['days'], skew_df['rr_25d'] * 100, 'o-', color='steelblue', linewidth=2, markersize=7)
ax.set_xlabel('Days to Expiry', fontsize=11)
ax.set_ylabel('25d Risk Reversal (%)', fontsize=11)
ax.set_title('25-Delta Risk Reversal', fontsize=12, fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# 25d Butterfly
ax = axes[2]
ax.plot(skew_df['days'], skew_df['bf_25d'] * 100, 'o-', color='forestgreen', linewidth=2, markersize=7)
ax.set_xlabel('Days to Expiry', fontsize=11)
ax.set_ylabel('25d Butterfly (%)', fontsize=11)
ax.set_title('25-Delta Butterfly', fontsize=12, fontweight='bold')

plt.suptitle('Skew Term Structure', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/skew_term_structure.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/skew_term_structure.png')

# 12. Arbitrage Check

In [ ]:
# Butterfly arbitrage per slice
k_test = np.linspace(-0.4, 0.4, 301)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Butterfly density (g function) per slice
ax = axes[0]
cmap_bf = plt.cm.viridis(np.linspace(0, 1, len(sorted_T)))

bf_results = []
for idx, T_key in enumerate(sorted_T):
    params = surface_params[T_key]
    g_vals, is_free = check_butterfly_arbitrage(
        k_test, params['a'], params['b'], params['rho'], params['m'], params['sigma']
    )
    ax.plot(k_test, g_vals, color=cmap_bf[idx], linewidth=1.5,
            label=f'{T_key*365:.0f}d ({"OK" if is_free else "VIOLATION"})')
    bf_results.append({'T_days': T_key * 365, 'butterfly_arb_free': is_free,
                       'min_g': float(np.min(g_vals))})

ax.axhline(y=0, color='crimson', linestyle='--', linewidth=1.5, label='No-arb boundary')
ax.set_xlabel('Log-Moneyness', fontsize=11)
ax.set_ylabel('g(k)', fontsize=11)
ax.set_title('Butterfly Arbitrage Density', fontsize=12, fontweight='bold')
ax.legend(fontsize=8, ncol=2)

# Right: Calendar arbitrage
ax = axes[1]
cal_result = check_calendar_arbitrage(surface_params)

if cal_result['is_arb_free']:
    ax.text(0.5, 0.5, 'No Calendar Arbitrage\nDetected',
            transform=ax.transAxes, ha='center', va='center',
            fontsize=16, color='forestgreen', fontweight='bold')
else:
    for v in cal_result['violations']:
        ax.bar(f"{v['T1']*365:.0f}d-{v['T2']*365:.0f}d",
               v['max_violation'] * 10000, color='crimson')
    ax.set_ylabel('Max Violation (total var, bps)', fontsize=11)

ax.set_title('Calendar Arbitrage Check', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figures/arbitrage_check.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/arbitrage_check.png')

print(f'\nCalendar arbitrage free: {cal_result["is_arb_free"]}')
if not cal_result['is_arb_free']:
    for v in cal_result['violations']:
        print(f'  Violation between T={v["T1"]*365:.0f}d and T={v["T2"]*365:.0f}d: '
              f'max_violation={v["max_violation"]:.6f}')

# 13. Smile Metrics Summary

In [ ]:
smile_metrics_df = pd.DataFrame(skew_metrics)
smile_metrics_df['atm_iv_pct'] = smile_metrics_df['atm_iv'] * 100
smile_metrics_df['rr_25d_pct'] = smile_metrics_df['rr_25d'] * 100
smile_metrics_df['bf_25d_pct'] = smile_metrics_df['bf_25d'] * 100

display_cols = ['days', 'atm_iv_pct', 'rr_25d_pct', 'bf_25d_pct', 'skew_slope']
print(smile_metrics_df[display_cols].to_string(index=False))

smile_metrics_df.to_csv('../results/tables/smile_metrics.csv', index=False)
print('\nSaved: results/tables/smile_metrics.csv')

## Key Findings

1. **Negative skew**: The equity volatility smile is systematically downward-sloping -- OTM puts are more expensive than OTM calls. This reflects demand for downside protection and the leverage effect (falling prices increase asset volatility).

2. **SVI calibration quality**: SVI fits each maturity slice with sub-100 bps RMSE and R-squared near 1.0, confirming it captures the smile shape well with just 5 parameters.

3. **Term structure**: Short-dated skew is steeper than long-dated skew. The ATM level may be upward or downward sloping depending on the current vol regime.

4. **Variance Risk Premium**: The difference between implied and realized vol is typically positive for SPY, meaning systematic sellers of options earn a premium over time.

5. **Arbitrage diagnostics**: The butterfly density $g(k)$ remains non-negative (no butterfly arbitrage), and total variance is non-decreasing across maturities (no calendar arbitrage) when the SVI parameters are well-calibrated.

6. **Realized vol estimators**: Yang-Zhang is the most efficient (lowest variance) estimator because it uses OHLC data; Parkinson uses only high-low and outperforms close-to-close; close-to-close is the simplest but noisiest.

In [ ]:
saved_files = [
    'results/figures/raw_market_ivs.png',
    'results/figures/realized_vs_implied.png',
    'results/tables/svi_calibration_summary.csv',
    'results/figures/vol_smiles.png',
    'results/figures/vol_surface_3d.png',
    'results/figures/vol_term_structure.png',
    'results/figures/skew_term_structure.png',
    'results/figures/arbitrage_check.png',
    'results/tables/smile_metrics.csv',
]

print('Files saved by this notebook:')
print('=' * 50)
for f in saved_files:
    print(f'  {f}')